In [ ]:
# Spark Master configuration
# Use "spark://spark-master:7077" inside Docker, or "local[*]" locally
import os
master = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")


### Spark Session & Profiler Setup (Lab 3)
Configuração da SparkSession com configurações de memória do worker/driver e profiler.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath("../src"))
sys.path.append("/workspace/lab03")
sys.path.append("/apps/lab03")

from pyspark.sql import SparkSession
try:
    from spark_profiler import SparkProfiler
except ImportError:
    SparkProfiler = None

spark_builder = (SparkSession.builder
    .master(master)
    .appName("CompPar-Project1-TaxiTrips")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "1536m")
    .config("spark.ui.showConsoleProgress", "false"))

spark = spark_builder.getOrCreate()
profiler = SparkProfiler(spark) if SparkProfiler else None

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores):", spark.sparkContext.defaultParallelism)


In [ ]:
import glob
from pyspark.sql import functions as sf
from pyspark.sql.functions import col, to_timestamp

# Localização flexível dos dados (funciona no Docker e localmente)
possible_data_paths = ["/data/project1", "/data", "../data", "./data"]
data_dir = next((d for d in possible_data_paths if os.path.exists(d)), "/data")
zones_path = os.path.join(data_dir, "taxi_zone_lookup.csv")
if not os.path.exists(zones_path):
    zones_path = "../data/taxi_zone_lookup.csv"

print(f"Reading zone lookup from: {zones_path}")
zones = (spark.read.option("delimiter", ",").option("header", True)
         .csv(zones_path))

# Cópias do lookup para pickup e dropoff com colunas renomeadas
pu = zones.select(sf.col("LocationID").cast("int").alias("PULocationID"),
                  sf.col("Borough").alias("PU_Borough"),
                  sf.col("Zone").alias("PU_Zone"),
                  sf.col("service_zone").alias("PU_service_zone"))

do = zones.select(sf.col("LocationID").cast("int").alias("DOLocationID"),
                  sf.col("Borough").alias("DO_Borough"),
                  sf.col("Zone").alias("DO_Zone"),
                  sf.col("service_zone").alias("DO_service_zone"))

# Leitura dos Parquet do dataset oficial (2016-2020) ou arquivos disponíveis
parquet_pattern = os.path.join(data_dir, "yellow_tripdata_*.parquet")
print(f"Reading trips from pattern: {parquet_pattern}")
raw_trips = spark.read.parquet(parquet_pattern)

# Cast types e projeção das colunas necessárias conforme enunciado
df_organized = raw_trips.select(
    to_timestamp(col("tpep_pickup_datetime")).alias("tpep_pickup_datetime"),
    to_timestamp(col("tpep_dropoff_datetime")).alias("tpep_dropoff_datetime"),
    col("PULocationID").cast("int").alias("PULocationID"),
    col("DOLocationID").cast("int").alias("DOLocationID"),
    col("passenger_count").cast("int").alias("passenger_count"),
    col("trip_distance").cast("double").alias("trip_distance"),
    col("fare_amount").cast("double").alias("fare_amount"),
    col("tip_amount").cast("double").alias("tip_amount"),
    col("payment_type").cast("int").alias("payment_type"),
    col("total_amount").cast("double").alias("total_amount")
)
print("Schema organizado:")
df_organized.printSchema()


### M1 — Data Ingestion & Cleaning
1. Filtrar registos inválidos segundo todos os 5 critérios do enunciado:
   - `fare_amount > 0`
   - `trip_distance > 0`
   - `passenger_count BETWEEN 1 AND 6`
   - `tpep_dropoff_datetime > tpep_pickup_datetime`
   - `PULocationID BETWEEN 1 AND 263` e `DOLocationID BETWEEN 1 AND 263`
2. **Calcular e reportar estatísticas anuais de limpeza** (número de registos removidos por critério).

In [ ]:
# Criação de flags booleanas para cada critério de remoção e ano da viagem
df_flagged = (df_organized
    .withColumn("year", sf.year("tpep_pickup_datetime"))
    .withColumn("drop_fare", (col("fare_amount") <= 0) | col("fare_amount").isNull())
    .withColumn("drop_distance", (col("trip_distance") <= 0) | col("trip_distance").isNull())
    .withColumn("drop_passenger", (col("passenger_count") <= 0) | (col("passenger_count") > 6) | col("passenger_count").isNull())
    .withColumn("drop_timestamps", (col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime")) | col("tpep_dropoff_datetime").isNull() | col("tpep_pickup_datetime").isNull())
    .withColumn("drop_locations", (~col("PULocationID").between(1, 263)) | (~col("DOLocationID").between(1, 263)) | col("PULocationID").isNull() | col("DOLocationID").isNull()))

# Relatório de estatísticas anuais de limpeza por critério (Requisito obrigatório do M1)
cleaning_stats = (df_flagged.groupBy("year").agg(
    sf.count("*").alias("total_records"),
    sf.sum(col("drop_fare").cast("long")).alias("removed_fare_le_0"),
    sf.sum(col("drop_distance").cast("long")).alias("removed_distance_le_0"),
    sf.sum(col("drop_passenger").cast("long")).alias("removed_invalid_passengers"),
    sf.sum(col("drop_timestamps").cast("long")).alias("removed_invalid_timestamps"),
    sf.sum(col("drop_locations").cast("long")).alias("removed_invalid_locations")
).orderBy("year"))

print("=== M1: Cleaning Statistics per Year ===")
cleaning_stats.show(truncate=False)

# DataFrame limpo (filtrando todas as viagens inválidas)
cleaned_df = (df_flagged.filter(
    (~col("drop_fare")) &
    (~col("drop_distance")) &
    (~col("drop_passenger")) &
    (~col("drop_timestamps")) &
    (~col("drop_locations"))
).drop("drop_fare", "drop_distance", "drop_passenger", "drop_timestamps", "drop_locations"))

# Cache do DataFrame limpo para acelerar etapas subsequentes (M2, M3, O1, O3)
cleaned_df = cleaned_df.cache()
print(f"Total de registos limpos válidos: {cleaned_df.count()}")


### M2 — Temporal Demand Analysis
Para cada triplo `(pickup_borough, day_of_week, hour_of_day)`, calcular:
- `total trip count`
- `mean fare_amount`
- `mean trip duration in minutes` (derivado de timestamps)
- `mean trip_distance`
Escrever o resultado particionado por borough em formato Parquet.

In [ ]:
# Junção broadcast com a tabela de zonas para obter o PU_Borough sobre os dados limpos
cleaned_with_pu = cleaned_df.join(sf.broadcast(pu), on="PULocationID", how="inner")

result2 = (cleaned_with_pu
    .withColumn("day_of_week", sf.dayofweek("tpep_pickup_datetime"))
    .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
    .withColumn("duration_min",
                (sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")) / 60.0)
    .groupBy(sf.col("PU_Borough").alias("pickup_borough"), "day_of_week", "hour_of_day")
    .agg(
        sf.count("*").alias("trip_count"),
        sf.avg("fare_amount").alias("mean_fare_amount"),
        sf.avg("duration_min").alias("mean_duration_min"),
        sf.avg("trip_distance").alias("mean_trip_distance")
    ))

output_m2_path = os.path.join(data_dir, "output", "borough_hour_stats")
(result2.write
      .mode("overwrite")
      .partitionBy("pickup_borough")
      .parquet(output_m2_path))
print(f"M2 escrito com sucesso em {output_m2_path}")


In [ ]:
spark.read.parquet(output_m2_path).orderBy("pickup_borough", "day_of_week", "hour_of_day").show(20)


### M3 — Zone-Level Statistics
1. Por cada zona de pickup e mês: calcular média e desvio padrão de `fare_amount` e `trip_distance`, e contagem de viagens.
2. Funções de janela (Window Functions):
   - Rank das zonas por volume mensal dentro de cada borough (`rank_in_borough`).
   - Share rolante de 3 meses de cada zona dentro do seu borough (`roll3_share`).
   - Flag de anomalia quando o volume do mês desvia mais de 1.5 desvios padrão da sua própria média rolante (`anomaly`).

In [ ]:
from pyspark.sql import Window

# 1. Agregação mensal por zona com métricas pedidas no enunciado
zone_monthly = (cleaned_with_pu
    .withColumn("month", sf.date_trunc("month", "tpep_pickup_datetime"))
    .withColumn("month_idx", sf.year("month") * 12 + sf.month("month"))
    .groupBy(sf.col("PU_Borough").alias("pickup_borough"), "PULocationID", "PU_Zone", "month_idx", "month")
    .agg(
        sf.count("*").alias("trips"),
        sf.avg("fare_amount").alias("mean_fare_amount"),
        sf.stddev("fare_amount").alias("std_fare_amount"),
        sf.avg("trip_distance").alias("mean_trip_distance"),
        sf.stddev("trip_distance").alias("std_trip_distance")
    ))

# 2. Especificação das janelas (Window Specs)
w_rank = Window.partitionBy("pickup_borough", "month_idx").orderBy(sf.desc("trips"))
w_roll_zone = Window.partitionBy("pickup_borough", "PULocationID").orderBy("month_idx").rangeBetween(-2, 0)
w_bor_month = Window.partitionBy("pickup_borough", "month_idx")

result3 = (zone_monthly
    .withColumn("rank_in_borough", sf.rank().over(w_rank))
    .withColumn("roll3_trips", sf.sum("trips").over(w_roll_zone))
    .withColumn("roll3_share", sf.col("roll3_trips") / sf.sum("roll3_trips").over(w_bor_month))
    .withColumn("roll_mean", sf.avg("trips").over(w_roll_zone))
    .withColumn("roll_std", sf.stddev("trips").over(w_roll_zone))
    .withColumn("anomaly",
                sf.coalesce(sf.abs(sf.col("trips") - sf.col("roll_mean")) > 1.5 * sf.col("roll_std"),
                            sf.lit(False))))

result3.orderBy("pickup_borough", "month_idx", "rank_in_borough").show(20, truncate=False)


### O1 — Fare Anomaly Detection (Opcional 1)
1. Para cada bucket `(PULocationID, hour_of_day)`, calcular média e $\sigma$ da tarifa por distância (`fare_amount / trip_distance`).
2. Identificar viagens anómalas que excedam $\text{mean} + 3\sigma$.
3. Comparar taxa de anomalias em dias úteis (weekday) vs fim-de-semana (weekend), agregadas por `payment_type` e `pickup_borough`.

In [ ]:
w_zone_hour = Window.partitionBy("PULocationID", "hour_of_day")

result4 = (cleaned_with_pu
    .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
    .withColumn("unit_fare", sf.col("fare_amount") / sf.col("trip_distance"))
    .withColumn("mean_unit_fare", sf.avg("unit_fare").over(w_zone_hour))
    .withColumn("sigma_unit_fare", sf.stddev("unit_fare").over(w_zone_hour))
    .withColumn("anomaly",
                sf.coalesce(sf.col("unit_fare") > (sf.col("mean_unit_fare") + 3 * sf.col("sigma_unit_fare")),
                            sf.lit(False)))
    .withColumn("day_type",
                sf.when(sf.dayofweek("tpep_pickup_datetime").isin(1, 7), "weekend").otherwise("weekday"))
    .groupBy(sf.col("PU_Borough").alias("pickup_borough"), "payment_type", "day_type")
    .agg(sf.count("*").alias("trips"),
         sf.sum(sf.col("anomaly").cast("int")).alias("anomalies"))
    .withColumn("anomaly_rate", sf.col("anomalies") / sf.col("trips"))
    .groupBy("pickup_borough", "payment_type")
    .pivot("day_type", ["weekday", "weekend"])
    .agg(sf.first("anomaly_rate"))
    .withColumn("weekend_minus_weekday", sf.col("weekend") - sf.col("weekday")))

result4.orderBy("pickup_borough", "payment_type").show(50)


### O3 — Demand Forecasting Features (Opcional 2)
1. Agregar viagens por `(PULocationID, pickup_date, day_of_week, hour_of_day)`.
2. Calcular média histórica e variância de viagens para cada tuplo `(PULocationID, day_of_week, hour_of_day)`.
3. **Self-join temporal com lag de 4 semanas (28 dias)** nas datas de pickup desfasadas.
4. Calcular o índice de procura (`current_count ÷ historical_mean`) e assinalar quando `demand_index > 2.0`.
5. Broadcast join com `taxi_zone_lookup` para anotar os resultados com `service_zone`, `Borough` e `Zone`.

In [ ]:
# 1. Contagem diária/horária por zona
daily_hourly = (cleaned_df
    .withColumn("pickup_date", sf.to_date("tpep_pickup_datetime"))
    .withColumn("day_of_week", sf.dayofweek("tpep_pickup_datetime"))
    .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
    .groupBy("PULocationID", "pickup_date", "day_of_week", "hour_of_day")
    .agg(sf.count("*").alias("trip_count")))

# 2. Estatísticas históricas por (PULocationID, day_of_week, hour_of_day)
hist_stats = (daily_hourly
    .groupBy("PULocationID", "day_of_week", "hour_of_day")
    .agg(
        sf.avg("trip_count").alias("historical_mean"),
        sf.variance("trip_count").alias("historical_variance")
    ))

# 3. Modelar o lag de 4 semanas (28 dias) através de self-join em datas desfasadas
curr = daily_hourly.alias("curr")
lag = daily_hourly.alias("lag")

joined_lag = curr.join(
    lag,
    (sf.col("curr.PULocationID") == sf.col("lag.PULocationID")) &
    (sf.col("curr.hour_of_day") == sf.col("lag.hour_of_day")) &
    (sf.col("lag.pickup_date") == sf.date_sub(sf.col("curr.pickup_date"), 28)),
    how="left"
).select(
    sf.col("curr.PULocationID").alias("PULocationID"),
    sf.col("curr.pickup_date").alias("pickup_date"),
    sf.col("curr.day_of_week").alias("day_of_week"),
    sf.col("curr.hour_of_day").alias("hour_of_day"),
    sf.col("curr.trip_count").alias("current_trip_count"),
    sf.coalesce(sf.col("lag.trip_count"), sf.lit(0)).alias("lag_4w_trip_count")
)

# 4. Derivar demand index (current ÷ historical mean) e flag > 2.0
# 5. Broadcast join com taxi_zone_lookup para anotar service_zone
result_o3 = (joined_lag
    .join(hist_stats, on=["PULocationID", "day_of_week", "hour_of_day"], how="inner")
    .withColumn("demand_index",
                sf.when(sf.col("historical_mean") > 0,
                        sf.col("current_trip_count") / sf.col("historical_mean")).otherwise(None))
    .withColumn("high_demand_flag", sf.coalesce(sf.col("demand_index") > 2.0, sf.lit(False)))
    .join(sf.broadcast(zones.select(
              sf.col("LocationID").cast("int").alias("PULocationID"),
              "service_zone", "Borough", "Zone")),
          on="PULocationID", how="left"))

result_o3.orderBy(sf.desc("demand_index")).show(30, truncate=False)
